# Reorder AstroFlame data and convert known units to CGS

This notebook reads `astroflame_unformatted_3.csv` and creates one CSV containing BASS and SAFFIRE-V observations:

- Key measurement fields (thickness, sample width, rib width, oxygen percentages, and fire-spread rate) appear first.
- Other fields that vary within an experiment follow, with experiment-constant details such as material and geometry at the end.

SAFFIRE-I/II/III rows are excluded. For SAFFIRE-V, the initial oxygen value is used when available; otherwise the reported `oxygen_percent` is copied to both `oxygen_percent` and `oxygen_initial_pct`. Thickness and rib width are converted from millimeters to centimeters, and fire-spread rate from mm/s to cm/s. Other source fields and all selected observation rows are retained.

In [4]:
from pathlib import Path
import pandas as pd

In [5]:
input_name = "astroflame_unformatted_3.csv"
candidate_dirs = [Path.cwd(), Path.cwd() / "elig-data"]
data_dir = next((path for path in candidate_dirs if (path / input_name).is_file()), None)
if data_dir is None:
    raise FileNotFoundError(f"Could not find {input_name} in the current folder or its elig-data subfolder.")

unformatted = data_dir / input_name
formatted = data_dir / "astroflame_formatted_cgs.csv"

In [6]:
source = pd.read_csv(unformatted)
required_columns = {
    "dataset",
    "experiment_id",
    "observation_id",
    "thickness_mm",
    "rib_width_mm",
    "fsr_mm_s",
    "oxygen_percent",
    "oxygen_initial_pct",
}
missing_columns = required_columns.difference(source.columns)
if missing_columns:
    raise ValueError(f"Missing required columns: {sorted(missing_columns)}")
if source.empty:
    raise ValueError("The input CSV contains no observations.")

keep_dataset = source["dataset"].str.upper().str.startswith("BASS", na=False) | source["dataset"].str.upper().isin({"SAFFIRE-V"})
df = source.loc[keep_dataset].copy()
if df.empty:
    raise ValueError("The input CSV contains no BASS or SAFFIRE-V observations.")

saffire = df["dataset"].str.upper().isin({"SAFFIRE-V"})
initial_oxygen = pd.to_numeric(df["oxygen_initial_pct"], errors="raise")
reported_oxygen = pd.to_numeric(df["oxygen_percent"], errors="raise")
saffire_oxygen = initial_oxygen.combine_first(reported_oxygen)
df.loc[saffire, "oxygen_percent"] = saffire_oxygen.loc[saffire]
df.loc[saffire, "oxygen_initial_pct"] = saffire_oxygen.loc[saffire]

for column in ["thickness_mm", "rib_width_mm", "fsr_mm_s"]:
    df[column] = pd.to_numeric(df[column], errors="raise")

df["thickness_cm"] = df.pop("thickness_mm") / 10
df["rib_width_cm"] = df.pop("rib_width_mm") / 10
df["fsr_cm_s"] = df.pop("fsr_mm_s") / 10

group_keys = ["dataset", "experiment_id"]
if df[group_keys].isna().any().any() or df["observation_id"].isna().any():
    raise ValueError("Dataset, experiment, and observation identifiers must not be blank.")

key_columns = [
    column
    for column in [
        "thickness_cm",
        "sample_width_cm",
        "rib_width_cm",
        "sample_length_cm",
        "oxygen_percent",
        "oxygen_initial_pct",
        "oxygen_final_pct",
        "fsr_cm_s",
    ]
    if column in df.columns
]
id_columns = ["dataset", "experiment_id", "observation_id"]
remaining_columns = [
    column for column in df.columns if column not in key_columns + id_columns
]
varies_within_experiment = (
    df.groupby(group_keys, dropna=False)[remaining_columns]
    .nunique(dropna=False)
    .gt(1)
    .any()
)
changing_columns = [
    column for column in remaining_columns if varies_within_experiment[column]
]
constant_columns = [
    column for column in remaining_columns if not varies_within_experiment[column]
]
df = df[key_columns + id_columns + changing_columns + constant_columns]

if len(df) != int(keep_dataset.sum()):
    raise AssertionError("Observation rows were lost while reordering columns.")
if df["observation_id"].tolist() != source.loc[keep_dataset, "observation_id"].tolist():
    raise AssertionError("Observation order or identifiers changed unexpectedly.")

df.to_csv(formatted, index=False)
print(f"Read and retained {len(df)} BASS/SAFFIRE-V observations; excluded {len(source) - len(df)} other rows.")
print(f"Wrote one CSV: {formatted.name}")
print(f"Key columns first: {key_columns}")
print(f"Other changing columns next: {changing_columns}")
print(f"Experiment-constant columns last: {constant_columns}")
print("Converted thickness_mm and rib_width_mm to centimeters, and fsr_mm_s to cm/s. SAFFIRE-V oxygen is mapped to both oxygen_percent and oxygen_initial_pct.")

df.head()

Read and retained 57 BASS/SAFFIRE-V observations; excluded 5 other rows.
Wrote one CSV: astroflame_formatted_cgs.csv
Key columns first: ['thickness_cm', 'sample_width_cm', 'rib_width_cm', 'sample_length_cm', 'oxygen_percent', 'oxygen_initial_pct', 'oxygen_final_pct', 'fsr_cm_s']
Other changing columns next: ['airflow_cm_s', 'notes', 'original_segment_number']
Experiment-constant columns last: ['material', 'geometry', 'flow_direction', 'oxygen_basis', 'burning_sides', 'measurement_method', 'source_location', 'pressure_kpa', 'rib_height_mm', 'base_thickness_mm', 'data_quality']
Converted thickness_mm and rib_width_mm to centimeters, and fsr_mm_s to cm/s. SAFFIRE-V oxygen is mapped to both oxygen_percent and oxygen_initial_pct.


,thickness_cm,sample_width_cm,rib_width_cm,sample_length_cm,oxygen_percent,oxygen_initial_pct,oxygen_final_pct,fsr_cm_s,dataset,experiment_id,...,geometry,flow_direction,oxygen_basis,burning_sides,measurement_method,source_location,pressure_kpa,rib_height_mm,base_thickness_mm,data_quality
0,0.1,2.2,NaN,NaN,NaN,22.2,21.9,0.0071,BASS-II PSI-25,M1,...,flat slab,opposed,segment oxygen unknown; test start/end recorde...,1.0,reported_table,NASA/TM-20210011385 Table 5.1 p.57,NaN,NaN,NaN,reported or source-derived; see measurement_me...
1,0.3,2.2,NaN,NaN,NaN,22.2,21.2,0.0052,BASS-II PSI-25,M2,...,flat slab,opposed,segment oxygen unknown; test start/end recorde...,1.0,reported_table,NASA/TM-20210011385 Table 5.1 p.57,NaN,NaN,NaN,reported or source-derived; see measurement_me...
2,0.3,2.2,NaN,NaN,NaN,22.2,21.2,0.0033,BASS-II PSI-25,M2,...,flat slab,opposed,segment oxygen unknown; test start/end recorde...,1.0,reported_table,NASA/TM-20210011385 Table 5.1 p.57,NaN,NaN,NaN,reported or source-derived; see measurement_me...
3,0.5,2.2,NaN,NaN,NaN,20.8,19.7,0.0029,BASS-II PSI-25,M3,...,flat slab,opposed,segment oxygen unknown; test start/end recorde...,1.0,reported_table,NASA/TM-20210011385 Table 5.1 p.57,NaN,NaN,NaN,reported or source-derived; see measurement_me...
4,0.5,2.2,NaN,NaN,NaN,20.8,19.7,0.0024,BASS-II PSI-25,M3,...,flat slab,opposed,segment oxygen unknown; test start/end recorde...,1.0,reported_table,NASA/TM-20210011385 Table 5.1 p.57,NaN,NaN,NaN,reported or source-derived; see measurement_me...
